In [0]:
%run ./lib/00_common

In [0]:
%run ./lib/03_logger_utils

In [0]:
import uuid

run_date = get_text_widget("run_date", "2025-04-01")
alert_email_to = get_text_widget("alert_email_to", "")
gmail_smtp_user = get_text_widget("gmail_smtp_user", "")
gmail_app_password = get_text_widget("gmail_app_password", "")

run_id = str(uuid.uuid4())

In [0]:


params_common = {
    "run_id": run_id,
    "run_date": run_date,
    "alert_email_to": alert_email_to,
    "gmail_smtp_user": gmail_smtp_user,
    "gmail_app_password": gmail_app_password
}

try:
    log_event(
        spark=spark,
        level="INFO",
        run_id=run_id,
        stage="orchestrate",
        dataset="pipeline_v2_pf",
        status="STARTED",
        message=f"Inicia pipeline ELT V2 pandas-first para run_date={run_date}"
    )

    dbutils.notebook.run(
        f"/Workspace/Users/juandavid2931@gmail.com/Python-course-for-data-engineering/04_ELT/01_Lab_Monitoreo_Logs_Orquestamiento/Shared/etl_lab/01_generate_raw_data_to_volume",
        0,
        {"run_date": run_date}
    )

    dbutils.notebook.run(
        f"/Workspace/Users/juandavid2931@gmail.com/Python-course-for-data-engineering/04_ELT/01_Lab_Monitoreo_Logs_Orquestamiento/Shared/etl_lab/02_bronze_ingest_pandas",
        0,
        params_common
    )

    dbutils.notebook.run(
        f"/Workspace/Users/juandavid2931@gmail.com/Python-course-for-data-engineering/04_ELT/01_Lab_Monitoreo_Logs_Orquestamiento/Shared/etl_lab/03_silver_transform_pandas",
        0,
        params_common
    )

    dbutils.notebook.run(
        f"/Workspace/Users/juandavid2931@gmail.com/Python-course-for-data-engineering/04_ELT/01_Lab_Monitoreo_Logs_Orquestamiento/Shared/etl_lab/04_gold_publish_pandas",
        0,
        params_common
    )

    dbutils.notebook.run(
        f"/Workspace/Users/juandavid2931@gmail.com/Python-course-for-data-engineering/04_ELT/01_Lab_Monitoreo_Logs_Orquestamiento/Shared/etl_lab/05_audit_versions",
        0,
        params_common
    )

    log_event(
        spark=spark,
        level="INFO",
        run_id=run_id,
        stage="orchestrate",
        dataset="pipeline_v2_pf",
        status="OK",
        message="Pipeline ELT V2 pandas-first completado correctamente"
    )

    print(f"PIPELINE V2 PF OK - run_id={run_id}")

except Exception as e:
    log_event(
        spark=spark,
        level="ERROR",
        run_id=run_id,
        stage="orchestrate",
        dataset="pipeline_v2_pf",
        status="FAILED",
        message=str(e),
        error_code="ORCH_001"
    )
    print(f"PIPELINE V2 PF FAILED - run_id={run_id}")
    raise